Purpose:
1. Generates all three Friedman datasets.
2. Splits into train/test sets.
3. Saves .npz files for individual model usage.

Angie McGraw

Last updated: 03/10/2026

### Generate Friedman 1-3 and Save to .npz's

In [1]:
# Verify packages
import sys
print(sys.executable)
import torch, sklearn, numpy
print(torch.__version__, sklearn.__version__)

/Users/angiemcgraw/miniforge3/envs/double_descent_env/bin/python



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/angiemcgraw/miniforge3/envs/double_descent_env/lib/python3.11/runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/angiemcgraw/miniforge3/envs/double_descent_env/lib/python3.11/runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "/Users/angiemcgraw/miniforge3/envs/double_descent_env/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/angiemcgraw/miniforge3/envs/double_descen

2.2.2 1.8.0


In [2]:
# Load the libraries
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.datasets import make_friedman1, make_friedman2, make_friedman3
from sklearn.model_selection import train_test_split

In [3]:
# Global experiment parameters
N_TRAIN = 100
N_TEST = 1000
NOISE = 1.0

seed_map = {
    "friedman1": 42,
    "friedman2": 52, 
    "friedman3": 62
}

# Will offset seeds for demo subset (seed), full dataset (seed + 100), train/test split (seed + 200)
# Demo subset is a sanity check.
# Offsetting seeds allows for distinct seeds to prevent overlapping X arrays for different datasets.

In [4]:
# Directory to save preliminary plots (i.e. plots for the Friedman datasets - initial view of the data)
#PLOT_DIR = "../preliminary_plots" # this is relative to /code

# Directory to save each Friedman dataset as individual .npz
DATA_DIR = "../data"

os.makedirs(DATA_DIR, exist_ok=True)

In [5]:
def generate_friedman_datasets(save_npz=True):
    """
    Generate train/test splits for all three Friedman datasets.

    Parameters
        save_npz:bool
            If True, saves each of the Friedman datasets as a separate .npz file to be loaded 
            in individual model scripts.

    Returns
        Dictionary of the Friedman datasets
        dict { 
            "friedman1": (X_train, X_test, y_train, y_test)
            "friedman2": (X_train, X_test, y_train, y_test)
            "friedman3": (X_train, X_test, y_train, y_test)
        }
    """
    n_total = N_TRAIN + N_TEST
    datasets = {}

    # Seed offsets
    DEMO_OFFSET = 0
    DATA_OFFSET = 100
    SPLIT_OFFSET = 200

    for name, func in [
        ("friedman1", make_friedman1),
        ("friedman2", make_friedman2),
        ("friedman3", make_friedman3),
    ]:

        base_seed = seed_map[name]
        
        # Demo subset check to match scikit-learn docs
        demo_seed = base_seed + DEMO_OFFSET
        if name == "friedman1":
            demo_X, demo_y = func(n_samples=5, n_features=5, random_state=demo_seed, noise=0.0)
        else:
            demo_X, demo_y = func(n_samples=5, random_state=demo_seed, noise=0.0)

        print(f"Demo {name.upper()} subset:")
        print("X:\n", demo_X)
        print("y:\n", demo_y)
        print("-"*50)
        
        # Generate full dataset (make_friedman1 n_features is required), i.e. 1100 datapoints
        # This is (n_total samples)
        data_seed = base_seed + DATA_OFFSET
        if name == "friedman1":
            X, y = func(n_samples=n_total, n_features=10, noise=NOISE, random_state=data_seed)
        else:
            X, y = func(n_samples=n_total, noise=NOISE, random_state=data_seed)

        # Inspect raw dataset before splitting
        print(f"Raw {name.upper()} dataset:")
        print(f"    X shape: {X.shape}, y shape: {y.shape}")
        print(f"    First 5 rows of X:\n{X[:5]}")
        print(f"    First 5 values of y: {y[:5]}")
        print("-"*50)

        # Train/test split
        split_seed = base_seed + SPLIT_OFFSET
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, train_size=N_TRAIN, test_size=N_TEST, random_state=split_seed
        )
        datasets[name] = (X_train, X_test, y_train, y_test)

        # Print shapes
        print(f"{name.upper()}:")
        print(f"    X_train: {X_train.shape}, y_train, {y_train.shape}")
        print(f"    X_test: {X_test.shape}, y_test, {y_test.shape}")
        print(f"    First 5 rows of X_train:\n{X_train[:5]}")
        print(f"    First 5 rows of y_train:\n{y_train[:5]}")
        print("-" * 40)

        # Save datasets as .npz files
        if save_npz:
            save_path = os.path.join(DATA_DIR, f"{name}.npz")
            np.savez(save_path, X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test)
            print(f"Saved {name} dataset to {save_path}.")

    return datasets

In [6]:
def generate(save_npz=True):
    """
    Generates all three Friedman datasets.
    """
    return generate_friedman_datasets(save_npz=save_npz)

In [7]:
# Generate all three Friedman datasets
datasets = generate(save_npz=True)

# Each Friedman dataset
X_train1, X_test1, y_train1, y_test1 = datasets["friedman1"]
X_train2, X_test2, y_train2, y_test2 = datasets["friedman2"]
X_train3, X_test3, y_train3, y_test3 = datasets["friedman3"]

Demo FRIEDMAN1 subset:
X:
 [[0.37454012 0.95071431 0.73199394 0.59865848 0.15601864]
 [0.15599452 0.05808361 0.86617615 0.60111501 0.70807258]
 [0.02058449 0.96990985 0.83244264 0.21233911 0.18182497]
 [0.18340451 0.30424224 0.52475643 0.43194502 0.29122914]
 [0.61185289 0.13949386 0.29214465 0.36636184 0.45606998]]
y:
 [16.83826156 12.51782504  5.86968919  7.53187897  9.45737165]
--------------------------------------------------
Raw FRIEDMAN1 dataset:
    X shape: (1100, 10), y shape: (1100,)
    First 5 rows of X:
[[0.90206152 0.55780754 0.65598471 0.83247141 0.19988419 0.12725426
  0.77143911 0.43228855 0.38528223 0.78364337]
 [0.78553162 0.16280861 0.11511333 0.33252155 0.60157345 0.87780234
  0.61487722 0.4240039  0.95784627 0.85408447]
 [0.77414528 0.87562841 0.07835524 0.03195872 0.70624246 0.55297016
  0.85526749 0.95410747 0.47497613 0.5481674 ]
 [0.73452535 0.53863186 0.74511416 0.57850545 0.17615075 0.7574693
  0.94823633 0.87774122 0.85551533 0.30142755]
 [0.10967141 0.906

### Load and verify .npz's

In [8]:
def load_friedman_npz():
    datasets = {}
    for name in ["friedman1", "friedman2", "friedman3"]:
        path = os.path.join(DATA_DIR, f"{name}.npz")
        with np.load(path) as data:
            X_train = data["X_train"]
            X_test = data["X_test"]
            y_train = data["y_train"]
            y_test = data["y_test"]
            datasets[name] = (X_train, X_test, y_train, y_test)

            print(f"{name.upper()} dataset loaded from {path}:")
            print(f"     X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
            print(f"     X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

            print(f"     First 5 rows of X_train:\n{X_train[:5]}")
            print(f"     First 5 values of y_train:\n{y_train[:5]}")
            print("-"*50)

    return datasets

In [9]:
if __name__ == '__main__':
    datasets = load_friedman_npz()

FRIEDMAN1 dataset loaded from ../data/friedman1.npz:
     X_train shape: (100, 10), y_train shape: (100,)
     X_test shape: (1000, 10), y_test shape: (1000,)
     First 5 rows of X_train:
[[0.1823516  0.59218462 0.70987552 0.20702041 0.73623359 0.74553734
  0.22898099 0.84860601 0.33494682 0.41453474]
 [0.48749886 0.11571604 0.97742188 0.68601084 0.5418102  0.97898505
  0.4428375  0.26002436 0.869821   0.73909465]
 [0.56680014 0.72780047 0.27478891 0.21331685 0.91188775 0.03964279
  0.93385481 0.53788413 0.9665781  0.39214212]
 [0.43991556 0.07962947 0.8174728  0.29409813 0.64540728 0.29196012
  0.98292513 0.90844349 0.4389489  0.25699777]
 [0.11165903 0.63238172 0.06644327 0.81722718 0.6062343  0.73880601
  0.01795117 0.1249949  0.182228   0.97675698]]
     First 5 values of y_train:
[ 8.82567883 17.65456368 17.58054052  8.98936113 17.65422451]
--------------------------------------------------
FRIEDMAN2 dataset loaded from ../data/friedman2.npz:
     X_train shape: (100, 4), y_train